In [2]:
# import libraries
import os
import re

# stopwords
STOPWORDS = set([
    "a","an","the","is","are","was","were","in","on","at","of","and","or","to"
])

# normalize word
def normalize(word):
    return word.lower()

# position class
class Position:
    def __init__(self, page_name, index):
        self.page_name = page_name
        self.index = index

# word entry
class WordEntry:
    def __init__(self, word):
        self.word = word
        self.positions = []

    def add_position(self, pos):
        self.positions.append(pos)

    def get_pages(self):
        return set(p.page_name for p in self.positions)

# page index
class PageIndex:
    def __init__(self):
        self.word_map = {}

    def add_word(self, word, pos):
        if word not in self.word_map:
            self.word_map[word] = WordEntry(word)
        self.word_map[word].add_position(pos)

# inverted index
class InvertedPageIndex:
    def __init__(self):
        self.index = {}

    def add_page(self, page_entry):
        for word, entry in page_entry.page_index.word_map.items():
            if word not in self.index:
                self.index[word] = WordEntry(word)

            for pos in entry.positions:
                self.index[word].add_position(pos)

    def get_pages(self, word):
        if word in self.index:
            return self.index[word].get_pages()
        return set()

    def get_positions(self, word, page):
        if word not in self.index:
            return []

        return [p.index for p in self.index[word].positions if p.page_name == page]

# page entry
class PageEntry:
    def __init__(self, page_name, text):
        self.page_name = page_name
        self.page_index = PageIndex()
        self.process(text)

    def process(self, text):
        words = re.findall(r'\w+', text)

        index = 0
        for w in words:
            w = normalize(w)

            if w in STOPWORDS:
                continue

            pos = Position(self.page_name, index)
            self.page_index.add_word(w, pos)
            index += 1

# search engine
class SearchEngine:
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path
        self.index = InvertedPageIndex()
        print("[INIT] dataset_path =", self.dataset_path)

    def add_page(self, page_name):
        webpages_path = os.path.join(self.dataset_path, "webpages")
        page_path = os.path.join(webpages_path, page_name)

        print("[ADD PAGE]", page_name)

        if not os.path.exists(page_path):
            print("OUTPUT: Page not found")
            return

        text = ""

        # if folder
        if os.path.isdir(page_path):
            for file in os.listdir(page_path):
                if file.startswith("."):
                    continue

                file_path = os.path.join(page_path, file)

                if not os.path.isfile(file_path):
                    continue

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        text += f.read().lower() + " "
                except:
                    pass

        # if single file
        else:
            with open(page_path, "r", encoding="utf-8") as f:
                text = f.read().lower()

        page_entry = PageEntry(page_name, text)
        self.index.add_page(page_entry)

# process actions
def process_actions(engine, actions_file):

    print("\n[INFO] Processing queries...\n")

    with open(actions_file, "r") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()

        if not line:
            continue

        parts = line.split()

        print("\n[QUERY]", line)

        # addPage
        if parts[0] == "addPage":
            engine.add_page(parts[1])
            print("OUTPUT: Page added")

        # find pages
        elif parts[0] == "queryFindPagesWhichContainWord":
            word = normalize(parts[1])

            pages = engine.index.get_pages(word)

            if pages:
                print("OUTPUT:", ", ".join(sorted(pages)))
            else:
                print("OUTPUT: No webpage contains word")

        # find positions
        elif parts[0] == "queryFindPositionsOfWordInAPage":
            word = normalize(parts[1])
            page = parts[2]

            positions = engine.index.get_positions(word, page)

            if positions:
                print("OUTPUT:", ", ".join(map(str, positions)))
            else:
                print("OUTPUT: Word not found in page")

        else:
            print("OUTPUT: Unsupported query")

# =========================
# main
# =========================

dataset_path = "/home/rajesh/CSL7100-Assignment4/Part2/dataset/Q2_websearch"

actions_file = os.path.join(dataset_path, "actions.txt")

print("Using dataset_path:", dataset_path)

engine = SearchEngine(dataset_path)

process_actions(engine, actions_file)

Using dataset_path: /home/rajesh/CSL7100-Assignment4/Part2/dataset/Q2_websearch
[INIT] dataset_path = /home/rajesh/CSL7100-Assignment4/Part2/dataset/Q2_websearch

[INFO] Processing queries...


[QUERY] addPage stack_datastructure_wiki
[ADD PAGE] stack_datastructure_wiki
OUTPUT: Page added

[QUERY] queryFindPagesWhichContainWord delhi
OUTPUT: No webpage contains word

[QUERY] queryFindPagesWhichContainWord stack
OUTPUT: stack_datastructure_wiki

[QUERY] queryFindPagesWhichContainWord wikipedia
OUTPUT: stack_datastructure_wiki

[QUERY] queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
OUTPUT: Word not found in page

[QUERY] queryFindPagesWhichContainWord allain
OUTPUT: No webpage contains word

[QUERY] addPage stack_cprogramming
[ADD PAGE] stack_cprogramming
OUTPUT: Page added

[QUERY] queryFindPagesWhichContainWord allain
OUTPUT: stack_cprogramming

[QUERY] queryFindPagesWhichContainWord C
OUTPUT: stack_cprogramming

[QUERY] queryFindPagesWhichContainWord C++
OUTPUT: No